# Avaliação do Sistema

## Aqui utilizarei 4 métricas de avaliação cada uma focada em algum ponto do RAG (Metricas para o Retriever e Generator)

| Métrica | Avalia | Como | Aplica-se a |
|---|---|---|---|
| F1 de abstenção (`f1_abstention`) | Recusou quando devia e respondeu quando devia | Python: precision e recall da recusa | todas |
| Recall de contexto (`context_recall`) | O retriever trouxe os documentos esperados | Python: TP / (TP + FN) por documento | perguntas com `ids_esperados` |
| Recall de valores (`values_recall`) | Os números esperados aparecem na resposta | Python: TP / (TP + FN) por valor | perguntas com `valores_esperados` |
| Faithfulness (`faithfulness`) | A resposta é fiel ao contexto (não inventa) | RAGAS: afirmações sustentadas / total de afirmações | respostas que não são recusa |

- **Entrada:** `eval/dataset.csv` (gabarito) e `eval/respostas_<modelo>.json` (respostas geradas pelo `generate.py`).
- **Saída:** `eval/resultados_<modelo>.csv`.
- Este notebook não gera respostas, só mede. As métricas em Python não gastam cota; só o Faithfulness (seção 5) chama o Gemini como juiz.

## 0. Carregar gabarito e respostas

In [ ]:
import json
import re

import pandas as pd

from config import LLM_MODEL
from generate import NOT_FOUND

dataset = pd.read_csv('../eval/dataset.csv')
with open(f'../eval/respostas_{LLM_MODEL}.json', encoding='utf-8') as arquivo:
    respostas = pd.DataFrame(json.load(arquivo))

# Junta gabarito e respostas pelo id; a pergunta já está no dataset, então sai do lado das respostas
df = dataset.merge(respostas.drop(columns='pergunta'), on='id', how='left')

# Conferir se todas as perguntas tem resposta gerada
sem_resposta = df[df['resposta'].isna()]['id'].tolist()
assert not sem_resposta, f'Sem resposta para {sem_resposta}: rode o generate.py até o fim.'

print(f'{len(df)} perguntas | modelo avaliado: {LLM_MODEL}')
df[['id', 'tipo', 'pergunta', 'resposta']]

## 1. Funções das métricas

- **Recall** = TP / (TP + FN): dos itens esperados, quantos apareceram
- **Precision** = TP / (TP + FP): dos itens que apareceram, quantos eram esperados
- **F1** = 2 · P · R / (P + R): média harmônica entre precision e recall

In [ ]:
def recall(tp, fn):
    # Dos itens esperados, quantos apareceram
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)


def precision(tp, fp):
    # Dos itens que apareceram, quantos eram esperados
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)


def f1(p, r):
    # Média harmônica: só é alta quando precision e recall são altos ao mesmo tempo
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

## 2. F1 de abstenção: `f1_abstention`

**Classe positiva:** "deve recusar" (perguntas do tipo `fora_escopo`). Uma resposta é recusa quando contém a frase `NOT_FOUND`.

| | Sistema recusou | Sistema respondeu |
|---|---|---|
| **Era fora de escopo** | TP: recusa certa | FN: fora de escopo que passou |
| **Era dentro do escopo** | FP: pergunta válida barrada | TN: resposta normal |

Usei F1 em vez de acurácia por causa do desequilíbrio: no dataset de exemplo são 2 perguntas fora de escopo contra 11 dentro. Um sistema que nunca recusa teria acurácia de 0,85 e F1 igual a 0.

In [ ]:
df['recusou'] = df['resposta'].str.contains(NOT_FOUND, regex=False)
df['deve_recusar'] = df['tipo'] == 'fora_escopo'
df['acerto_escopo'] = df['recusou'] == df['deve_recusar']

tp = (df['deve_recusar'] & df['recusou']).sum()     # recusa certa
fn = (df['deve_recusar'] & ~df['recusou']).sum()    # fora de escopo que passou
fp = (~df['deve_recusar'] & df['recusou']).sum()    # pergunta válida barrada
tn = (~df['deve_recusar'] & ~df['recusou']).sum()   # resposta normal

p_escopo = precision(tp, fp)
r_escopo = recall(tp, fn)
f1_abstention = f1(p_escopo, r_escopo)
acuracia = (tp + tn) / len(df)

print(f'TP={tp} FN={fn} FP={fp} TN={tn}')
print(f'Precision: {p_escopo:.2f} | Recall: {r_escopo:.2f} | F1: {f1_abstention:.2f} | Acurácia (comparação): {acuracia:.2f}')

In [ ]:
print('Fora de escopo que passaram (FN):', df[df['deve_recusar'] & ~df['recusou']]['id'].tolist())
print('Perguntas válidas barradas (FP):', df[~df['deve_recusar'] & df['recusou']]['id'].tolist())

# Linhas: o que era esperado | colunas: o que o sistema fez
pd.crosstab(df['deve_recusar'], df['recusou'], rownames=['deve recusar'], colnames=['recusou'])

## 3. Recall de contexto: `context_recall` (retriever)

Para cada pergunta com `ids_esperados`:
- **TP:** documento esperado que está nas `fontes` (documentos enviados à LLM)
- **FN:** documento esperado que não está nas `fontes`

Não se aplica às analíticas (quem responde é a tool) nem às de fora de escopo (medidas pelo `f1_abstention`).

In [ ]:
df['context_recall'] = float('nan')  # NaN = métrica não se aplica à pergunta

for i, linha in df[df['ids_esperados'].notna()].iterrows():
    esperados = linha['ids_esperados'].split(';')
    tp = sum(1 for doc in esperados if doc in linha['fontes'])  # esperado que foi recuperado
    fn = len(esperados) - tp                                     # esperado que ficou de fora
    df.loc[i, 'context_recall'] = recall(tp, fn)
    print(f"{linha['id']}: TP={tp} FN={fn} | esperados={esperados}")

print(f"\ncontext_recall médio: {df['context_recall'].mean():.2f}")
df[df['ids_esperados'].notna()][['id', 'tipo', 'ids_esperados', 'fontes', 'context_recall']]

## 4. Recall de valores: `values_recall` (acerto dos números)

Para cada pergunta com `valores_esperados`:
- **TP:** valor esperado que aparece na resposta
- **FN:** valor esperado ausente ou errado na resposta

Os números da resposta precisam ser lidos nos dois formatos: brasileiro (`232.890,85`) e americano (`232,890.85`), com uma tolerância pequena para arredondamento.

> Um número **a mais** e errado na resposta seria um FP, e o recall não pune FP. Esse caso é coberto pelo Faithfulness.

In [ ]:
def numeros(texto):
    # Cada sequência de dígitos (com '.' ou ',' no meio) é lida nos dois formatos;
    # basta um deles bater com o valor esperado
    encontrados = []
    for token in re.findall(r'\d[\d.,]*\d|\d', texto):
        formato_br = token.replace('.', '').replace(',', '.')  # '232.890,85' → '232890.85'
        formato_us = token.replace(',', '')                    # '232,890.85' → '232890.85'
        for candidato in [formato_br, formato_us]:
            try:
                encontrados.append(float(candidato))
            except ValueError:
                pass  # ex.: '1.2.3' não vira número em nenhum formato
    return encontrados


def valor_na_resposta(valor, numeros_resposta, tolerancia=0.01):
    # O sinal é ignorado: a resposta pode escrever "queda de 33,89%" em vez de "-33,89%"
    return any(abs(abs(valor) - n) <= tolerancia for n in numeros_resposta)


numeros('total de US$ 232.890,85 e queda de -33.89%')  # teste rápido

In [ ]:
df['values_recall'] = float('nan')

for i, linha in df[df['valores_esperados'].notna()].iterrows():
    esperados = [float(v) for v in str(linha['valores_esperados']).split(';')]
    numeros_resposta = numeros(linha['resposta'])
    tp = sum(1 for valor in esperados if valor_na_resposta(valor, numeros_resposta))
    fn = len(esperados) - tp
    df.loc[i, 'values_recall'] = recall(tp, fn)
    print(f"{linha['id']} ({linha['tipo']}): TP={tp} FN={fn} | esperados={esperados}")

analiticas = df['tipo'] == 'analitica'
print(f"\nvalues_recall médio | analíticas: {df[analiticas]['values_recall'].mean():.2f}"
      f" | demais: {df[~analiticas]['values_recall'].mean():.2f}")
df[df['valores_esperados'].notna()][['id', 'tipo', 'valores_esperados', 'values_recall']]

## 5. Faithfulness com RAGAS: `faithfulness` (generator)

Mede se a resposta inventa algo que não está no contexto. O juiz (Gemini) faz duas etapas para cada resposta:
1. Quebra a resposta em afirmações.
2. Verifica se cada afirmação é sustentada pelo contexto.

**faithfulness** = afirmações sustentadas / total de afirmações

- O contexto inclui os documentos **e o resultado das tools** (coluna `contexto`, montada no `generate.py`). Sem isso, os números das respostas analíticas pareceriam inventados.
- Recusas ficam fora (NaN): não há afirmações para verificar.
- Usei RAGAS porque ele já implementa as duas etapas com prompts testados e leitura estruturada da resposta do juiz; as métricas simples ficaram em Python.
- **Limitação:** o juiz é o mesmo modelo que gerou as respostas, o que pode deixar a avaliação mais condescendente. A nota também pode variar um pouco entre execuções.
- **Custo:** cerca de 2 chamadas ao Gemini por resposta (≈ 2 × nº de respostas que não são recusa).

In [ ]:
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

from config import client, JUDGE_MODEL

juiz = llm_factory(JUDGE_MODEL, provider='google', client=client)
faithfulness = Faithfulness(llm=juiz)
print('Juiz:', JUDGE_MODEL)

In [ ]:
import time

TENTATIVAS = 3
ESPERA = 30  # s entre tentativas: o 503 é sobrecarga temporária do modelo no plano gratuito

# Cria a coluna só na primeira execução: rodar a célula de novo refaz apenas as perguntas que falharam
if 'faithfulness' not in df.columns:
    df['faithfulness'] = float('nan')

pendentes = df[~df['recusou'] & df['faithfulness'].isna()]
print(f'{len(pendentes)} respostas para avaliar')

for i, linha in pendentes.iterrows():
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            # await: no notebook o RAGAS precisa da versão assíncrona (ascore); o score() síncrono não roda no Jupyter
            resultado = await faithfulness.ascore(
                user_input=linha['pergunta'],
                response=linha['resposta'],
                retrieved_contexts=linha['contexto'],
            )
            df.loc[i, 'faithfulness'] = resultado.value
            print(f"{linha['id']}: {resultado.value:.2f}")
            break  # deu certo: sai das tentativas e vai para a próxima pergunta
        except Exception as erro:
            print(f"{linha['id']}: tentativa {tentativa} falhou ({str(erro)[:60]}...)")
            if tentativa < TENTATIVAS:
                time.sleep(ESPERA)

faltando = df[~df['recusou'] & df['faithfulness'].isna()]['id'].tolist()
print(f"\nfaithfulness médio: {df['faithfulness'].mean():.2f} | sem nota: {faltando or 'nenhuma'}")
df[~df['recusou']][['id', 'tipo', 'faithfulness']]

## 6. Resultado final

In [ ]:
metricas = ['acerto_escopo', 'context_recall', 'values_recall', 'faithfulness']

# Média por tipo; NaN = a métrica não se aplica àquele tipo
resumo = df.groupby('tipo')[metricas].mean()
resumo.loc['TOTAL'] = df[metricas].mean()
print(f'f1_abstention (todas as perguntas): {f1_abstention:.2f}')

colunas = ['id', 'tipo', 'pergunta', 'resposta', 'recusou', 'acerto_escopo', 'context_recall', 'values_recall', 'faithfulness']
df[colunas].to_csv(f'../eval/resultados_{LLM_MODEL}.csv', index=False)

resumo.round(2)